# Explore linear fluence  law 

## Zhou 2023: https://www.sciencedirect.com/science/article/pii/S1369800123001919
## Shen 2010


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import curve_fit


In [ ]:

class fluencePowerLaw:
    def __init__(self, c_0, phi_0):
        self.c_0 = c_0
        self.phi_0 = phi_0

    def eval(self, fluence):
        return 1 - self.c_0 * np.log(1 + fluence/self.phi_0)
    
    def plot(self, min_fluence, max_fluence):
        fluences = np.range(min_fluence, max_fluence)
        return plt.plot(self.eval(fluences))


In [ ]:
# Fit model to csv of data
# Required columns: 'fluence (e/cm^2)', 'V_oc', 'I_SC', 'P_max'
# TODO-TD: we could make a dataclass to wrap a dataframe?
# Required vs optional columns

df = pd.read_csv("./Wang2021.csv")

def model(x, c, x0):
    return 1 - c * np.log(1 + x / x0)


wang_2021_results = {}
for (ptype, energy), grp in df.groupby(['particle type', 'energy (MeV)']):
    
    x = grp['fluence (e/cm^2)'].values
    if len(x) > 2:
        print(f"{ptype} {energy} MeV")
        ks = ['V_oc', 'I_SC', 'P_max']
        wang_2021_results[f'{energy} MeV {ptype} GaInP/GaAs/Ge'] = dict.fromkeys(ks)

        for name in ks:
            y = grp[name].values
            # initial guesses matter here — x0 must stay > 0 (log domain)
            # so give curve_fit bounds to keep it well-behaved
            p0 = [1.0, 1.0]              # [c, x0] starting guess
            bounds = ([-np.inf, 1e-8], [np.inf, np.inf])  # x0 > 0

            popt, pcov = curve_fit(model, x, y, p0=p0, bounds=bounds)
            c_fit, x0_fit = popt
            perr = np.sqrt(np.diag(pcov))  # 1-sigma parameter errors

            print(f" {name} c  = {c_fit:.6g} ± {perr[0]:.2g}")
            print(f" {name} phi0 = {x0_fit:.6g} ± {perr[1]:.2g}")

            # goodness of fit
            y_pred = model(x, *popt)
            ss_res = np.sum((y - y_pred)**2)
            ss_tot = np.sum((y - y.mean())**2)
            r2 = 1 - ss_res / ss_tot
            print(f"\tR squared = {r2:.4f}")
            wang_2021_results[f'{energy} MeV {ptype} GaInP/GaAs/Ge'][name] = {
                'Cp':c_fit,
                'Phi_0':x0_fit
            }


In [ ]:

plt.scatter(
    0.018,
    3.251e9,
    label='5 MeV p+ InGaAs Eg = 1eV'
)

plt.scatter(
    0.0747,
    3.148e10,
    label='5 MeV p+ InGaAs Eg = 0.7eV'
)
plt.scatter(
    0.213,
    0.61e9,
    label='3 MeV p+ InGaAs Eg = 0.74 eV'
)

plt.scatter(
    0.168,
    0.92e9,
    label='1MeV e- InGaAs Eg = 0.74 eV'
)

for k, v in wang_2021_results.items():
    plt.scatter(
        v['V_oc']['Cp'],
        v['V_oc']['Phi_0'],
        label=k
    )

plt.title('V_OC')
plt.xlabel('Cp')
plt.ylabel('Phi_0')
plt.yscale('log')
plt.grid()
plt.legend()

In [ ]:

plt.scatter(
    0.0261,
    1.902e9,
    label='5 MeV p+ InGaAs Eg = 1eV'
)

plt.scatter(
    0.0241,
    1.904e10,
    label='5 MeV p+ InGaAs Eg = 0.7eV'
)

for k, v in wang_2021_results.items():
    plt.scatter(
        v['I_SC']['Cp'],
        v['I_SC']['Phi_0'],
        label=k
    )

plt.title('I_SC')
plt.xlabel('Cp')
plt.ylabel('Phi_0')
plt.yscale('log')
plt.grid()
plt.legend()

In [ ]:

plt.scatter(
    0.0667,
    3.678e9,
    label='5 MeV p+ InGaAs Eg = 1eV'
)

plt.scatter(
    0.0984,
    1.312e10,
    label='5 MeV p+ InGaAs Eg = 0.7eV'
)

for k, v in wang_2021_results.items():
    plt.scatter(
        v['P_max']['Cp'],
        v['P_max']['Phi_0'],
        label=k
    )

plt.title('P_max')
plt.xlabel('Cp')
plt.ylabel('Phi_0')
plt.yscale('log')
plt.grid()
plt.legend()